# Week 2 Presentation Brief — Variation B
## ☢️ Radioactive Waste: The Fukushima Legacy
**SCIE1500 — Semester 2, 2026 | Group 2 — presenting in Week 3**

> Work through all parts during the Week 2 lab. Your **10-minute Week 3 presentation** should cover: the problem, your model, key results, and policy implications.
> Do not show raw code in your presentation — show graphs and equations.
> **What to submit:** upload your completed presentation slides (PDF or PowerPoint) to the LMS after your presentation.


---
## 📋 Scenario

![Radioactive decay curves for Cs-137 and Sr-90](../images/W2B_radioactive_fukushima.svg)

You are advising the **Japanese government** on long-term storage of radioactive waste from the Fukushima disaster. The primary isotope of concern is **Caesium-137** (half-life: **30.17 years**).

- Initial container activity: **500 TBq** (terabecquerels)
- **Regulatory threshold:** Activity must drop below **10 TBq** before the container can be reclassified as "low-level waste" (enabling cheaper storage).

---
## 🎯 Your Task

| Part | Topic | Time |
|------|-------|------|
| A | Build decay models in two forms; find reclassification year | ~20 min |
| B | Add Sr-90 isotope; find when activities equalise | ~15 min |
| C | Analyse storage costs over 150 years | ~10 min |
| D | Apply carbon dating to connect half-life to an archaeological question | ~10 min |

### Getting started — what does the model look like?

Radioactive decay is exponential *decrease*. The activity of Cs-137 at time $t$ (years after 2011) is:
$$A(t) = 500 \times e^{-\lambda t}, \quad \lambda = \frac{\ln 2}{30.17}$$

The decay constant $\lambda$ is always positive; the **negative sign** in the exponent is what makes the function decrease. The larger $\lambda$ is, the faster the isotope decays.

Run the setup cell to define $\lambda$ and inspect its value.

Setup — run first.

In [ ]:
# Setup — run first
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

A0_cs     = 500        # initial Cs-137 activity (TBq)
T_half_cs = 30.17      # half-life in years
threshold = 10         # reclassification threshold (TBq)

lambda_cs = np.log(2) / T_half_cs
print("lambda_cs =", lambda_cs)    # raw value — per year
print("A0 =", A0_cs, "TBq")
print("Reclassification threshold:", threshold, "TBq")

---
## Part A: Modeling Radioactive Decay (~20 min)

We'll write the Cs-137 decay model in two equivalent forms, verify they agree, then solve for the year when the container activity drops below the regulatory threshold.

### A.1 — Two equivalent forms of the decay model

Just as bacteria can be modelled using either the doubling form or the $e^{kt}$ form, radioactive decay can be written as:
- **Half-life form:** $A(t) = A_0 \times (0.5)^{t/T_{\text{half}}}$ — each half-life, multiply by 0.5
- **Decay-constant form:** $A(t) = A_0 \times e^{-\lambda t}$ — continuous decay at rate $\lambda$

Run the cell and confirm both forms give the same activity at 0, 1, 2, and 3 half-lives.

Two forms of the Cs-137 decay model.

In [ ]:
# A.1 — Two forms of the Cs-137 decay model
def A_cs_v1(t): return A0_cs * (0.5)**(t / T_half_cs)     # half-life form
def A_cs_v2(t): return A0_cs * np.exp(-lambda_cs * t)      # decay-constant form

print("Verifying both forms agree:")
for label, t in [("t=0", 0), ("1 half-life", T_half_cs),
                 ("2 half-lives", 2*T_half_cs), ("100 years", 100)]:
    print(f"  {label:>15} ({t:>6.1f} y):  form1 = {A_cs_v1(t):.3f} TBq   form2 = {A_cs_v2(t):.3f} TBq")

### A.2 — When does the activity reach the reclassification threshold?

We need to find $t$ such that $A(t) = 10$ TBq. Starting from the decay-constant form:
$$500 \, e^{-\lambda t} = 10 \implies t = \frac{-\ln(10/500)}{\lambda}$$

After finding $t$, we add it to 2011 to get the calendar year of reclassification.

Time to reach the 10 TBq threshold.

In [ ]:
# A.2 — Time to reach the 10 TBq threshold
t_threshold = -np.log(threshold / A0_cs) / lambda_cs
print(f"Time to reach {threshold} TBq: {t_threshold:.1f} years")
print(f"Reclassification year: {2011 + t_threshold:.0f}")

t_vals = np.linspace(0, 250, 500)
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(t_vals, A_cs_v1(t_vals), "steelblue", lw=2, label="Cs-137 activity")
ax.axhline(threshold, color="red",     ls="--", lw=1.5, label=f"Threshold ({threshold} TBq)")
ax.axvline(t_threshold, color="orange", ls=":",  lw=1.5, label=f"Threshold at {t_threshold:.0f} yrs")
ax.set_yscale("log")
ax.set_xlabel("Years since 2011")
ax.set_ylabel("Activity (TBq, log scale)")
ax.set_title("Cs-137 Decay — Fukushima Waste Container")
ax.legend()
plt.tight_layout()
plt.show()

✏️ **Group discussion A:**
1. In what calendar year can the container be reclassified? How many half-lives is that?
2. Why does activity never actually reach zero?

```python
# Answers:
# 1. Number of half-lives to reclassification:
print(f"Number of half-lives: {t_threshold / T_half_cs:.2f}")
# 2. Why never zero:
#    ...
```

---
## Part B: Multiple Isotopes (~15 min)

The waste also contains **Strontium-90**: half-life **28.8 years**, initial activity **200 TBq**. Both isotopes are present simultaneously and decay independently. Because Cs-137 starts with higher activity but the two half-lives are similar, their activity curves will cross at some point.

### B.1 — Adding Sr-90 to the model

We define a separate decay function for Sr-90 using its own $\lambda$, then plot both curves together. Notice that despite Cs-137 starting higher, the curves converge — we'll find exactly when they cross in B.2.

Sr-90 decay model.

In [ ]:
# B.1 — Sr-90 decay model
A0_sr     = 200
T_half_sr = 28.8
lambda_sr = np.log(2) / T_half_sr

def A_sr(t): return A0_sr * np.exp(-lambda_sr * t)

print(f"Cs-137: λ = {lambda_cs:.5f} /yr  (T½ = {T_half_cs} y)")
print(f"Sr-90:  λ = {lambda_sr:.5f} /yr  (T½ = {T_half_sr} y)")
print(f"Sr-90 decays faster than Cs-137? {lambda_sr > lambda_cs}")

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(t_vals, A_cs_v1(t_vals), "steelblue",  lw=2, label=f"Cs-137 (500 TBq, T½ = {T_half_cs} y)")
ax.plot(t_vals, A_sr(t_vals),    "darkorange", lw=2, label=f"Sr-90  (200 TBq, T½ = {T_half_sr} y)")
ax.axhline(threshold, color="red", ls="--", label="Reclassification threshold")
ax.set_yscale("log")
ax.set_xlabel("Years since 2011")
ax.set_ylabel("Activity (TBq)")
ax.set_title("Fukushima Waste: Cs-137 vs Sr-90 Decay")
ax.legend()
plt.tight_layout()
plt.show()

### B.2 — Finding the crossover analytically

When do the two activities equalise? We set $A_{\text{Cs}}(t) = A_{\text{Sr}}(t)$ and solve:
$$500 \, e^{-\lambda_{\text{Cs}} t} = 200 \, e^{-\lambda_{\text{Sr}} t}$$

Taking logarithms of both sides and rearranging gives $t = \ln(500/200) / (\lambda_{\text{Cs}} - \lambda_{\text{Sr}})$.

Crossover time: when do Cs-137 and Sr-90 have equal activity?

In [ ]:
# B.2 — Crossover time: when do Cs-137 and Sr-90 have equal activity?
delta_lambda = lambda_cs - lambda_sr
t_equal = np.log(A0_cs / A0_sr) / delta_lambda

print(f"Equal activity at t = {t_equal:.1f} years  (calendar year {2011 + t_equal:.0f})")
print(f"Activity at crossover: Cs = {A_cs_v2(t_equal):.2f} TBq   Sr = {A_sr(t_equal):.2f} TBq")

✏️ **Group discussion B:**
1. Cs-137 starts with more activity but has a *slightly longer* half-life than Sr-90. Why do they eventually equalise?
2. After the crossover, which isotope dominates total activity? Does the plot confirm this?

```python
# Answers:
# 1. Why they equalise:
#    ...
# 2. Which dominates post-crossover:
#    ...
```

---
## Part C: Storage Cost Analysis (~10 min)

High-level nuclear waste is expensive to store. Once the Cs-137 activity drops below 10 TBq, the container can be reclassified, cutting storage costs substantially.

- High-level storage: ¥50 million/year
- Low-level storage: ¥5 million/year
- Planning horizon: 150 years from 2011

We'll calculate total costs with and without reclassification.

In [ ]:
# C.1 — Storage cost over 150 years
yrs_high = t_threshold           # high-level until threshold
yrs_low  = 150 - t_threshold     # low-level after

cost_high       = yrs_high * 50
cost_low        = yrs_low  * 5
cost_total      = cost_high + cost_low
cost_no_reclas  = 150 * 50       # counterfactual: never reclassified

print(f"High-level period:  {yrs_high:.1f} years")
print(f"Low-level period:   {yrs_low:.1f} years")
print()
print(f"High-level cost:  ¥{cost_high:,.0f}M")
print(f"Low-level cost:   ¥{cost_low:,.0f}M")
print(f"Total:            ¥{cost_total:,.0f}M")
print(f"Saving vs always-high: ¥{cost_no_reclas - cost_total:,.0f}M")

✏️ **Policy summary (write as a group):**

Write 3–4 sentences for policymakers. Include:
- When reclassification occurs and the cost saving it enables
- One key assumption in your model that could change the answer
- A concrete recommendation

```
SUMMARY:
...
```

---
## Part D: Carbon Dating — Connecting Half-Life to Archaeology (~10 min)

The same exponential decay mathematics applies across wildly different timescales. Carbon-14 has a half-life of 5,730 years — compared to Cs-137's 30 years. Yet the equation is identical; only the numbers change. Here we use C-14 to date a fossil and compare the timescales.

In [ ]:
# D.1 — Carbon dating: fossil has 12.5% of original C-14
T_half_c14  = 5730
lambda_c14  = np.log(2) / T_half_c14

fraction     = 0.125    # 12.5% = (0.5)^3 → exactly 3 half-lives
t_fossil     = -np.log(fraction) / lambda_c14
n_halflives  = np.log(1 / fraction) / np.log(2)

print(f"12.5% = (0.5)^{n_halflives:.0f}  →  {n_halflives:.0f} half-lives")
print(f"Age: {t_fossil:.0f} years  ({n_halflives:.0f} × {T_half_c14} = {int(n_halflives*T_half_c14)} years)")

t_c14 = np.linspace(0, 45000, 400)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(t_c14 / 1000, 100 * np.exp(-lambda_c14 * t_c14), "purple", lw=2)
ax.axhline(12.5, color="green",  ls="--", label="12.5% remaining")
ax.axvline(t_fossil / 1000, color="orange", ls=":", label=f"{t_fossil:.0f} years")
ax.set_xlabel("Time (thousand years)")
ax.set_ylabel("C-14 remaining (%)")
ax.set_title("Carbon-14 Decay — Archaeological Dating")
ax.legend()
plt.tight_layout()
plt.show()

✏️ **Connecting the concepts:**

Compare the timescales: Cs-137 (30 y half-life), Sr-90 (29 y), C-14 (5,730 y). All three use the formula $A(t) = A_0 e^{-\lambda t}$.

What determines whether a decay process is "fast" or "slow" in practical terms? How does $\lambda$ capture this?

```
Your answer:
...
```

---
## ✅ Presentation Checklist (Week 3, 10 minutes)

1. **The Problem** (~2 min): What question were you answering?
2. **The Model** (~3 min): Why exponential decay? Show the equation and what $\lambda$ means.
3. **Key Results** (~3 min): Reclassification year, cost savings, crossover point — show the two-isotope plot.
4. **Implications** (~2 min): What should Japanese policymakers take away?

> **Tips:** Contrast Cs-137 and C-14 timescales — it makes the math feel real. "30 years vs 5,730 years — same equation, wildly different consequences." 